In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

data = pd.read_csv('dataset/training_data.csv', header=None, delimiter="\t", names=["labels", "text"])
data_test = pd.read_csv("dataset/testing_data.csv", header=None, delimiter="\t", names=["labels", "text"])

n_folds = 5

kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
data_test.head()

,labels,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes
3,2,mi school sends welcome back packet warning ki...
4,2,u.n. seeks 'massive' aid boost amid rohingya '...


In [30]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5658.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
data_encoded = model.encode(data.text.values)
data_test_encoded = model.encode(data_test.text.values)

In [32]:
# model - the model that will be fitted
# MODEL_NAME - with what name to save it 
# X - the data that is used in the KFold
# y - labels for the X
# test_data - the data that is used for LB 
def fit_predict_save(model, MODEL_NAME, X, y, test_data):
    d_e = X.copy()
    d_t_e = test_data.copy()
    oof = np.zeros((X.shape[0]))
    pred = np.zeros((test_data.shape[0]))

    for i, (train_index, test_index) in  enumerate(kf.split(d_e)):
        print("FOLD: ", i, "TRAIN:", train_index, "TEST:", test_index)
        X_train, X_test = d_e[train_index], d_e[test_index]
        y_train = y[train_index]

        model.fit(X_train, y_train)

        oof[test_index] = model.predict_proba(X_test)[:, 1] # for every index of test in every fold find the probability of 1, and as a result we will have all indexes covered
        pred += model.predict_proba(d_t_e)[:, 1] # find probability of 1 for every data_test item __5_times__ (n_folds)

    pred /= n_folds # since we summed 5 (n_folds) times, normalize it 

    # Save 
    pd.DataFrame({MODEL_NAME: pred}).to_csv(f"results/{MODEL_NAME}.csv", index=False)
    pd.DataFrame({MODEL_NAME + "_oof": oof}).to_csv(f"results/{MODEL_NAME}_oof.csv", index=False)

In [33]:
from sklearn.linear_model import LogisticRegression

fit_predict_save(LogisticRegression(), "LogisticRegression", data_encoded, data["labels"], data_test_encoded)


FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    9    11    13 ... 34140 34143 34151]


In [34]:
from sklearn.naive_bayes import MultinomialNB
import numpy as np

# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded), np.min(data_test_encoded)])
data_positive_encoded = data_encoded - min_val
data_test_positive_encoded = data_test_encoded - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB", data_positive_encoded, data["labels"], data_test_positive_encoded)


FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    9    11    13 ... 34140 34143 34151]


Much more on left bottom

In [35]:
from sklearn.ensemble import RandomForestClassifier

#fit_predict_save(RandomForestClassifier(), "RandomForestClassifier", data_encoded, data["labels"], data_test_encoded)

In [36]:
from sklearn.ensemble import GradientBoostingClassifier
 
#fit_predict_save(GradientBoostingClassifier(), "GradientBoostingClassifier", data_encoded, data["labels"], data_test_encoded)

In [37]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
data_encoded_bow = vectorizer.fit_transform(data["text"])
data_test_encoded_bow = vectorizer.transform(data_test["text"])

In [38]:
fit_predict_save(LogisticRegression(), "LogisticRegression_BOW", data_encoded_bow, data["labels"], data_test_encoded_bow)


# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded_bow), np.min(data_test_encoded_bow)])
data_positive_encoded = data_encoded_bow - min_val
data_test_positive_encoded = data_test_encoded_bow - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB_BOW", data_positive_encoded, data["labels"], data_test_positive_encoded)
#fit_predict_save(RandomForestClassifier(), "RandomForestClassifier_BOW", data_encoded_bow, data["labels"], data_test_encoded_bow)

FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    9    11    13 ... 34140 34143 34151]
FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 3

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
data_encoded = vectorizer.fit_transform(data["text"])
data_test_encoded = vectorizer.transform(data_test["text"])

In [40]:
fit_predict_save(LogisticRegression(), "LogisticRegression_TFIDF", data_encoded, data["labels"], data_test_encoded)


# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded), np.min(data_test_encoded)])
data_positive_encoded = data_encoded - min_val
data_test_positive_encoded = data_test_encoded - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB_TFIDF", data_positive_encoded, data["labels"], data_test_positive_encoded)
fit_predict_save(RandomForestClassifier(), "RandomForestClassifier_TFIDF", data_encoded, data["labels"], data_test_encoded)

FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    9    11    13 ... 34140 34143 34151]
FOLD:  0 TRAIN: [    1     2     3 ... 34148 34149 34151] TEST: [    0     4     6 ... 34122 34139 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    3    29    30 ... 34132 34144 34147]
FOLD:  2 TRAIN: [    0     1     2 ... 34149 34150 34151] TEST: [    5    14    15 ... 34141 34146 34148]
FOLD:  3 TRAIN: [    0     3     4 ... 34148 34150 34151] TEST: [    1     2    10 ... 34142 34145 34149]
FOLD:  4 TRAIN: [    0     1     2 ... 34148 3

KeyboardInterrupt: 